# Neural Network from Scratch
**Author:** Gaurang Bhatia

This notebook demonstrates the end-to-end process of building, training, and evaluating a fully connected neural network built entirely from scratch using only NumPy. By avoiding deep learning frameworks, we can transparently observe how data flows through forward passes, how error propagates backward using the chain rule, and how weights dynamically update via stochastic gradient descent with momentum. The following cells will walk through the mathematical foundations, network architecture, and evaluation metrics on the MNIST handwritten digit dataset.

First, we import the necessary modules from our `src` directory and set a random seed to ensure our results are fully reproducible.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Ensure we can import from src
sys.path.append(os.path.abspath('..'))
from src.utils import load_mnist, train_val_split, one_hot_encode
from src.optimizers import SGDMomentum
from src.network import NeuralNetwork
from src.layers import DenseLayer

np.random.seed(42)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## The Math
At the heart of the neural network lies **backpropagation**, which computes the gradient of the loss function with respect to every weight in the network. It relies on the **chain rule** of calculus:
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial A} \cdot \frac{\partial A}{\partial Z} \cdot \frac{\partial Z}{\partial W} $$

**Forward Pass:**
- Calculate pre-activation: $Z = W \cdot A_{prev} + b$
- Apply non-linearity: $A = \text{activation}(Z)$
- Cache both $Z$ and $A_{prev}$ to use later during the gradient calculations.

**Backward Pass:**
- Determine local error gradient: $dZ = dA * \text{activation}'(Z)$
- Update parameters: $dW = \frac{1}{m} dZ \cdot A_{prev}^T$ and $db = \frac{1}{m} \sum dZ$
- Propagate error to previous layer: $dA_{prev} = W^T \cdot dZ$

Below, we load the MNIST dataset and visualize a random 5x5 grid of training samples to understand what our model will be learning.

In [ ]:
X_train_full, Y_train_full, X_test, Y_test = load_mnist()
X_train, X_val, Y_train, Y_val = train_val_split(X_train_full, Y_train_full, val_size=0.1, seed=42)

Y_train_oh = one_hot_encode(Y_train, n_classes=10)
Y_val_oh = one_hot_encode(Y_val, n_classes=10)

# Visualize 25 samples
plt.figure(figsize=(10, 10))
random_indices = np.random.choice(X_train.shape[1], 25, replace=False)
for i, idx in enumerate(random_indices):
    plt.subplot(5, 5, i + 1)
    img = X_train[:, idx].reshape(28, 28)
    plt.imshow(img, cmap='gray')
    plt.title(f"Label: {Y_train[idx]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## Network Architecture
The network is a 3-layer dense architecture:
```text
[Input: 784] ---> [Dense 1: 128] ---> [Dense 2: 64] ---> [Output: 10]
   Flatten          ReLU Activ.          ReLU Activ.      Softmax Activ.
```
In the following code block, we instantiate the `NeuralNetwork` with `SGDMomentum`, attach our fully connected layers, and print a summary of the parameters.

In [ ]:
optimizer = SGDMomentum(learning_rate=0.01, beta=0.9)
nn = NeuralNetwork(optimizer)

nn.add_layer(DenseLayer(n_inputs=784, n_neurons=128, activation='relu'))
nn.add_layer(DenseLayer(n_inputs=128, n_neurons=64, activation='relu'))
nn.add_layer(DenseLayer(n_inputs=64, n_neurons=10, activation='softmax'))

total_params = 0
print("-" * 55)
print(f"{'Layer':<15} | {'Output Shape':<12} | {'Parameters'}")
print("-" * 55)
for i, layer in enumerate(nn.layers):
    params = layer.W.size + layer.b.size
    total_params += params
    print(f"DenseLayer_{i+1:<4} | ({layer.W.shape[0]:<10}) | {params:,}")
print("-" * 55)
print(f"Total Parameters: {total_params:,}")

With our dataset loaded and our network initialized, it's time to train. We will run mini-batch gradient descent for 20 epochs, printing verbose statistics so we can monitor loss and accuracy dynamically.

In [ ]:
nn.train(X_train, Y_train_oh, X_val, Y_val_oh, epochs=20, batch_size=32, verbose=True)

The training is complete. Now we can visualize the learning process by plotting the training vs validation loss curves and accuracy curves side-by-side.

In [ ]:
epochs = range(1, 21)

plt.figure(figsize=(14, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(epochs, nn.loss_history, label='Train Loss', color='blue')
plt.plot(epochs, nn.val_loss_history, label='Val Loss', color='orange')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(epochs, nn.accuracy_history, label='Train Accuracy', color='blue')
plt.plot(epochs, nn.val_accuracy_history, label='Val Accuracy', color='orange')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

To get a deeper understanding of our network's blind spots, we run predictions on the test set and plot a Confusion Matrix. This heatmap will highlight which digits our model frequently confuses.

In [ ]:
predictions = nn.predict(X_test)
cm = confusion_matrix(Y_test, predictions)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix — Test Set")
plt.colorbar()
tick_marks = np.arange(10)
plt.xticks(tick_marks, tick_marks)
plt.yticks(tick_marks, tick_marks)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

Finally, let's take a look at 20 random samples from the test set to visually compare the true labels against our model's predictions. The titles are colored green for correct predictions and red for incorrect ones.

In [ ]:
plt.figure(figsize=(12, 10))
plt.suptitle("Sample Predictions", fontsize=16)

random_indices = np.random.choice(X_test.shape[1], 20, replace=False)

for i, idx in enumerate(random_indices):
    plt.subplot(4, 5, i + 1)
    img = X_test[:, idx].reshape(28, 28)
    true_label = Y_test[idx]
    pred_label = predictions[idx]
    
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    
    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"Pred: {pred_label} | True: {true_label}", color=color, fontsize=10)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## What I Learned
- **Backpropagation Intuition**: Building the chain rule manually demystifies the 'black box' of neural networks, proving that learning is just recursive calculus through layers.
- **The Necessity of Xavier Initialization**: Starting with basic standard normal weights causes gradients to vanish, proving that properly scaled variance is essential for deep networks to learn.
- **Mini-Batch Dynamics**: Processing full batches takes too long, and singular samples are too erratic. Mini-batches strike the perfect balance of computational efficiency and gradient stability.
- **Momentum Accelerates Convergence**: Implementing the velocity vectors in SGD smooths out the noisy learning steps and significantly boosts test accuracy compared to standard SGD.
- **Mathematical Simplifications**: Realizing that the complex combination of softmax and categorical cross-entropy elegantly collapses into the derivative `Y_pred - Y_true` is mathematically profoundly satisfying.